upload wav to /data/wav. and tpv file in /data/ipa.tpv

In [2]:
# ==========================================
# 1. INSTALLATION & MOUNTING
# ==========================================
!pip install -q triton
!pip install -q git+https://github.com/sustcsonglin/flash-linear-attention
!pip install -q encodec torchaudio pandas transformers epitran

import os
import torch
import torch.nn as nn
import pandas as pd
import torchaudio
# Check if Triton is available
try:
    import triton
    print("✅ Triton installed successfully")
except ImportError:
    print("❌ Triton still missing")

import epitran
import IPython.display as ipd
from google.colab import drive
from torch.optim import AdamW
from encodec import EncodecModel
from fla.layers import GatedLinearAttention

# Mount your Google Drive
drive.mount('/content/drive')

# ==========================================
# 2. PATH CONFIGURATION
# ==========================================
# After adding the shortcut, find the exact folder name in your 'My Drive'
# Replace 'YourFolderShortcutName' with the actual name of that folder
DRIVE_FOLDER = '/content/drive/MyDrive/ne_fe_voice/'
WAV_DIR = os.path.join(DRIVE_FOLDER, 'wav')
METADATA_PATH = os.path.join(DRIVE_FOLDER, 'ipa.tsv')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Triton installed successfully
Mounted at /content/drive


In [3]:

try:
    # Load the TSV (using the 3rd column for phonemes)
    df = pd.read_csv(METADATA_PATH, sep='\t', header=None, names=['filename', 'text', 'phonemes'])

    # Clean: remove the '/' marks from the phonemes
    df['phonemes'] = df['phonemes'].str.strip('/')

    # Build a character map (Vocab) from the IPA symbols in your file
    # This ensures your model understands the specific symbols used in Nepali
    all_chars = sorted(list(set("".join(df['phonemes'].astype(str).tolist()))))
    char_to_id = {char: i + 2 for i, char in enumerate(all_chars)}
    char_to_id['<PAD>'] = 0
    char_to_id['<UNK>'] = 1

    VOCAB_SIZE = len(char_to_id)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"✅ Metadata loaded: {len(df)} rows found.")
    print(f"✅ Vocab Size: {VOCAB_SIZE} (IPA symbols mapped).")
    print(f"✅ Device: {device}")
    print("\nNote: Epitran skipped. Using IPA from metadata for training/testing.")

except Exception as e:
    print(f"❌ Error: {e}")
    print("Check if 'ne_fe_voice' folder is shared or if 'ipa.tsv' exists inside it.")

✅ Metadata loaded: 2064 rows found.
✅ Vocab Size: 45 (IPA symbols mapped).
✅ Device: cuda

Note: Epitran skipped. Using IPA from metadata for training/testing.


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("⚠️ WARNING: No GPU found. Training will be extremely slow.")

# Initialize EnCodec (The "Voice" generator)
codec_model = EncodecModel.encodec_model_24khz().to(device)
codec_model.set_target_bandwidth(6.0)

class DeepNepaliGLA(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, n_layers=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            GatedLinearAttention(mode='chunk', hidden_size=d_model, num_heads=n_heads)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        # Added a non-linear head to help with complex audio token mapping
        self.audio_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 1024)
        )

    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            # Catching the extra output from GLA
            output, *rest = layer(x)
            # RESIDUAL CONNECTION: Crucial for speech
            x = x + output
        x = self.norm(x)
        return self.audio_head(x)

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Downloading: "https://dl.fbaipublicfiles.com/encodec/v0/encodec_24khz-d7cc33bc.th" to /root/.cache/torch/hub/checkpoints/encodec_24khz-d7cc33bc.th


100%|██████████| 88.9M/88.9M [00:00<00:00, 132MB/s]


In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import torchaudio
from torch.optim import AdamW
from encodec import EncodecModel
from fla.layers import GatedLinearAttention

# 1. SETUP DEVICE (CPU/GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. DEFINE MODEL (With Residual Connections to stop the chirping)
class NepaliVoiceGLA(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, n_layers=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        # Positional encoding to help model know time steps after stretching
        self.pos_emb = nn.Parameter(torch.zeros(1, 5000, d_model))
        self.layers = nn.ModuleList([
            GatedLinearAttention(mode='chunk', hidden_size=d_model, num_heads=n_heads)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.audio_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 1024)
        )

    def forward(self, x, target_len=None):
        # x: [batch, text_len]
        x = self.embedding(x) # [batch, text_len, d_model]
        
        # Stretch embeddings to target_len BEFORE passing to GLA
        if target_len is not None and target_len > x.size(1):
            x = x.transpose(1, 2)
            x = torch.nn.functional.interpolate(x, size=target_len, mode='linear', align_corners=False)
            x = x.transpose(1, 2)
            
        seq_len = x.size(1)
        x = x + self.pos_emb[:, :seq_len, :]
        
        for layer in self.layers:
            output, *rest = layer(x)
            x = x + output  # Residual Connection is key!
        x = self.norm(x)
        return self.audio_head(x)

# 3. INITIALIZE MODELS
# Note: VOCAB_SIZE and char_to_id must be defined from your previous dataframe cell
# Fallback if VOCAB_SIZE is not defined (for testing)
if 'VOCAB_SIZE' not in globals():
    VOCAB_SIZE = 100
model = NepaliVoiceGLA(VOCAB_SIZE).to(device)

codec_model = EncodecModel.encodec_model_24khz().to(device)
codec_model.set_target_bandwidth(6.0)

# 4. TRAINING FUNCTION
def train_power(epochs=100):
    optimizer = AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    model.train()

    subset = df.head(5) # Start small to force a voice to emerge
    print(f"🚀 Starting Power Training on {len(subset)} samples...")

    for epoch in range(epochs):
        total_loss, count = 0, 0
        for idx, row in subset.iterrows():
            fname = str(row['filename']).strip()
            if not fname.endswith('.wav'): fname += '.wav'
            path = os.path.join(WAV_DIR, fname)
            if not os.path.exists(path): continue

            # Load & Resample
            wav, sr = torchaudio.load(path)
            if sr != 24000:
                wav = torchaudio.transforms.Resample(sr, 24000)(wav)

            # Move to device
            wav = wav.to(device)
            with torch.no_grad():
                # Get the ground truth audio tokens
                target = codec_model.encode(wav.unsqueeze(0))[0][0][:, 0, :]

            # Text to tokens
            tokens = torch.tensor([char_to_id.get(c, 1) for c in row['phonemes']]).unsqueeze(0).to(device)

            optimizer.zero_grad()
            # Pass target_len to stretch before GLA!
            logits = model(tokens, target_len=target.size(1))

            loss = criterion(logits.reshape(-1, 1024), target.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            count += 1

        if count > 0 and (epoch + 1) % 10 == 0:
            sample_pred = torch.argmax(logits[0], dim=-1)
            unique_sounds = len(torch.unique(sample_pred))
            print(f"Epoch {epoch+1:03d} | Loss: {total_loss/count:.4f} | Unique Sounds: {unique_sounds}")

# RUN IT
# train_power(100)


Using device: cuda
🚀 Starting Power Training on 5 samples...
Epoch 010 | Loss: 2.1999 | Unique Sounds: 80
Epoch 020 | Loss: 1.1019 | Unique Sounds: 140
Epoch 030 | Loss: 0.7902 | Unique Sounds: 149
Epoch 040 | Loss: 0.6175 | Unique Sounds: 162
Epoch 050 | Loss: 0.5704 | Unique Sounds: 161
Epoch 060 | Loss: 0.4763 | Unique Sounds: 169
Epoch 070 | Loss: 0.4000 | Unique Sounds: 175
Epoch 080 | Loss: 0.3615 | Unique Sounds: 172
Epoch 090 | Loss: 0.4352 | Unique Sounds: 165
Epoch 100 | Loss: 0.7062 | Unique Sounds: 160


In [ ]:
def generate_nepali(index=0, stretch=10):
    model.eval()
    row = df.iloc[index]
    input_ids = torch.tensor([char_to_id.get(c, 1) for c in row['phonemes']]).unsqueeze(0).to(device)

    with torch.no_grad():
        target_len = int(input_ids.size(1) * stretch)
        logits = model(input_ids, target_len=target_len)

        pred_tokens = torch.argmax(logits, dim=-1)

        # Build EnCodec format
        dummy_codes = torch.zeros((1, 8, pred_tokens.size(1)), dtype=torch.long).to(device)
        dummy_codes[:, 0, :] = pred_tokens
        wav_out = codec_model.decode([(dummy_codes, None)])

    print(f"Text: {row['text']}")
    return wav_out.cpu().squeeze().numpy()

# Try with stretch=10 to get a reasonable audio length
audio = generate_nepali(index=0, stretch=10)
ipd.display(ipd.Audio(audio, rate=24000))


Text: दीपा धामीको जन्म सुदूरपश्चिम नेपालको बझाङ जिल्लामा भएको हो
